# The analysis workflow, end to end

This notebook walks through Genesis Lab's whole research pipeline on a **small demo experiment**:

1. **Define** an experiment as data (a spec), with its hypothesis written down *first*
2. **Run** it: every condition × seed, in parallel
3. **Load** the results into the Parquet lake
4. **Query** them with **SQL** (DuckDB)
5. **Analyse** with **pandas**: one row per run, descriptive statistics
6. **Estimate**: confidence intervals, effect size, practical significance
7. **Test**: permutation and sign tests, cross-checked with SciPy
8. **Chart**: individual runs vs summaries
9. **Export**: files that can be reanalysed without Genesis Lab

**How to run it:** open it in VS Code, choose the kernel from this project's `.venv`, then *Run All*. It takes well under a minute. The demo writes to `data/notebook-demo/` (set the `GENESIS_NOTEBOOK_DIR` environment variable to change that) and never touches real experiment data.

**Before committing**, use *Clear All Outputs*: a test checks the committed notebook has none.

The demo (E990) is a teaching example, not a formal experiment: few seeds, short runs.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio
from scipy import stats

from genesis.analytics.analysis import (
    bootstrap_ci,
    describe_by,
    paired_effect,
    practical_significance,
    query_df,
    wilson_interval,
)
from genesis.analytics.batch import run_experiment
from genesis.analytics.significance import paired_permutation_test, sign_test
from genesis.analytics.spec import PROJECT_ROOT, ExperimentSpec
from genesis.lake.export import export_experiment, verify_export
from genesis.report import dashboard_figures
from genesis.report.dashboard_data import load_experiment

pio.renderers.default = "plotly_mimetype"  # charts render natively in VS Code and Jupyter

DEMO = Path(os.environ.get("GENESIS_NOTEBOOK_DIR", PROJECT_ROOT / "data" / "notebook-demo"))
LAKE = DEMO / "lake"
DEMO

## 1 · Define the experiment, and the hypothesis, before running anything

**Question:** does halving the food supply lower the population an evolving ecosystem settles at?

**Hypothesis (stated now, before the runs):** settled population with scarce food (regrowth 1.0) is lower than with plentiful food (2.0), paired by seed.

**Smallest effect of interest (SESOI):** 10 organisms. A difference smaller than that wouldn't matter for anything we'd do next.

**Metric:** each run's mean population over ticks 1000–1499. **Unit of analysis:** the run.

The experiment itself is data: a base config, conditions (named config changes) and seeds. Loading it builds and validates every run's config before anything runs.

In [ ]:
spec = ExperimentSpec.model_validate(
    {
        "experiment_id": "E990",
        "title": "Demo: plentiful vs scarce food",
        "definition": "notebooks/analysis-workflow.ipynb",
        "base_config": "configs/evolution.yaml",
        "base_changes": {"run.max_ticks": 1500},
        "seeds": {"first": 300, "count": 6},
        "conditions": [
            {"label": "plenty", "changes": {}},
            {"label": "scarce", "changes": {"food.regrowth_per_tick": 1.0}},
        ],
        "output": str(DEMO / "runs"),
    }
)
configs = spec.configs()  # every run's config, validated now
print(f"{len(configs)} runs planned:", sorted(configs)[:4], "...")

## 2 · Run it, and load the results into the lake

`run_experiment` skips runs that are already complete (with the same config), runs the rest in parallel worker processes, then validates and converts every run folder into Parquet files in the lake. Re-running this cell is cheap: everything is already done.

In [ ]:
outcomes = run_experiment(spec, rerun_stale=True, lake=LAKE)
print(f"ran {len(outcomes)} runs now; {len(configs) - len(outcomes)} were already complete")

## 3 · Ask questions in SQL

`query_df` runs SQL against the lake's views and returns a pandas **DataFrame**. `completed_runs` leaves out failed runs automatically.

The metric is computed in SQL: **join** each run's tick rows to its run (on `run_id`, always), keep the window, and **group by** run.

In [ ]:
per_run = query_df(
    """
    SELECT r.run_id, r.condition, r.seed, avg(t.population) AS settled_population
    FROM ticks AS t
    JOIN completed_runs AS r ON r.run_id = t.run_id
    WHERE r.experiment_id = 'E990' AND t.tick BETWEEN 1000 AND 1499
    GROUP BY r.run_id, r.condition, r.seed
    ORDER BY r.condition, r.seed
""",
    LAKE,
)
per_run

A **window function** compares each tick with the one before it without collapsing rows: here, the biggest one-tick population drop in each run.

In [ ]:
query_df(
    """
    SELECT condition, seed, max(drop) AS biggest_one_tick_drop
    FROM (
        SELECT r.condition, r.seed,
               lag(t.population) OVER (PARTITION BY t.run_id ORDER BY t.tick) - t.population AS drop
        FROM ticks t JOIN completed_runs r USING (run_id)
        WHERE r.experiment_id = 'E990'
    )
    GROUP BY condition, seed ORDER BY condition, seed
""",
    LAKE,
)

## 4 · Describe each condition

One row per run, so these statistics describe **runs**, never organisms. `sd` is the *sample* standard deviation (dividing by n − 1): we're estimating how much runs vary in general.

In [ ]:
describe_by(per_run, "condition", "settled_population").round(2)

## 5 · Estimate: how precisely do we know the medians?

A **bootstrap confidence interval** resamples the runs with replacement thousands of times. Seeded, so it gives the same interval every time.

In [ ]:
rng = np.random.default_rng(990)
for condition, part in per_run.groupby("condition"):
    ci = bootstrap_ci(part["settled_population"].to_numpy(), np.median, rng=rng)
    print(f"{condition:<7} median settled population {ci}   (n = {len(part)} runs)")

For a yes/no outcome (did a run go extinct?) the interval is a **Wilson interval**. With zero extinctions it still has a sensible upper bound: "none seen" isn't "never happens".

In [ ]:
extinct = query_df(
    "SELECT count(*) FILTER (WHERE stop_reason = 'extinction') AS k, count(*) AS n "
    "FROM completed_runs WHERE experiment_id = 'E990'",
    LAKE,
).iloc[0]
print(
    f"{extinct.k} of {extinct.n} runs went extinct; 95% Wilson interval for the rate: "
    f"{wilson_interval(int(extinct.k), int(extinct.n))}"
)

## 6 · Effect size and practical significance

Seeds are shared between conditions (same founders, same starting food), so compare them **in pairs**. The effect is the paired difference *scarce − plenty*, with a bootstrap interval. Then compare that interval with the SESOI of 10 organisms stated in section 1.

In [ ]:
wide = per_run.pivot(index="seed", columns="condition", values="settled_population")
effect = paired_effect(wide["scarce"], wide["plenty"], rng=np.random.default_rng(991))
print(f"mean difference (scarce - plenty): {effect.mean_difference:.1f} organisms")
print(f"95% CI: [{effect.ci.low:.1f}, {effect.ci.high:.1f}]")
print(f"relative to plentiful: {effect.relative_to_control:+.0%}")
print(f"Cohen's d_z: {effect.cohens_dz:.2f}")
print("practical significance vs SESOI 10:", practical_significance(effect.ci, 10))

## 7 · Test: could chance alone explain it?

The **paired permutation test** flips the sign of each seed's difference in every possible way (2⁶ = 64 here) to build chance's distribution. The **sign test** uses only directions. Both are cross-checked with SciPy's independent implementations.

With 6 pairs, the smallest p-value possible is 2/64 ≈ 0.031: few replicates limit how strong the evidence can ever be.

In [ ]:
differences = (wide["scarce"] - wide["plenty"]).to_numpy()
ours = paired_permutation_test(differences)
scipy_perm = stats.permutation_test(
    (wide["scarce"].to_numpy(), wide["plenty"].to_numpy()),
    lambda a, b, axis: np.mean(a - b, axis=axis),
    permutation_type="samples",
    n_resamples=np.inf,
)
sign = sign_test(differences)
print(
    f"permutation p: ours {ours.p_value:.4f} (exact: {ours.exact}), SciPy {scipy_perm.pvalue:.4f}"
)
print(
    f"sign test: {sign.positive} up / {sign.negative} down, p = {sign.p_value:.4f} "
    f"(SciPy binomtest {stats.binomtest(sign.positive, sign.positive + sign.negative).pvalue:.4f})"
)

**Reading it together:** the test says whether the difference is distinguishable from zero; the interval says how big it is and how precisely we know it; the SESOI says whether that size matters. Report all three.

## 8 · Chart: individual runs and summaries

The same chart the experiment dashboard uses: each faint line is one run, and the bold line is each condition's median across runs, with the middle 50% of runs shaded. (`uv run genesis dashboard E990 --lake data/notebook-demo/lake` builds the full page.)

In [ ]:
data = load_experiment("E990", LAKE, window=(1000, 1499))
dashboard_figures.population_figure(data)

## 9 · Export, verify, and reload without Genesis Lab

An export holds one file per table, a README describing every column, and a manifest with checksums. Reloading uses plain pandas only.

In [ ]:
export_dir = DEMO / "export"
manifest = export_experiment("E990", export_dir, lake=LAKE)
print("files:", list(manifest["files"]), "| verify:", verify_export(export_dir) or "intact")

ticks = pd.read_parquet(export_dir / "ticks.parquet")
runs = pd.read_parquet(export_dir / "runs.parquet")
window = ticks[ticks["tick"].between(1000, 1499)]
reloaded = window.groupby("run_id")["population"].mean().rename("settled_population")
check = (
    runs.set_index("run_id")[["condition", "seed"]]
    .join(reloaded)
    .sort_values(["condition", "seed"])
)
same = np.allclose(check["settled_population"].to_numpy(), per_run["settled_population"].to_numpy())
print("reanalysis from the export matches section 3:", same)

## What to try next

- Change the **seeds** to `{"first": 300, "count": 12}` and re-run: which numbers change a lot (interval widths, the smallest possible p) and which barely move (the medians)?
- Add a third condition, `{"label": "famine", "changes": {"food.regrowth_per_tick": 0.4}}`: does anything go extinct? Look at the Wilson interval.
- The SQL exercises in `docs/sql-exercises.md` (increment 3.9) ask questions of the real experiments, E001–E003.